# Exercise 5.1
Use the print_sampled_tokens function to print the sampling frequencies of the
softmax probabilities scaled with the temperatures shown in figure 5.14. How often
is the word pizza sampled in each case? Can you think of a faster and more accurate
way to determine how often the word pizza is sampled?

In [ ]:
import sys
sys.path.append('../01_main-chapter-code')

In [ ]:
import torch

In [ ]:
vocab = { 
    "closer": 0,
    "every": 1, 
    "effort": 2, 
    "forward": 3,
    "inches": 4,
    "moves": 5, 
    "pizza": 6,
    "toward": 7,
    "you": 8,
} 

inverse_vocab = {v: k for k, v in vocab.items()}

# Suppose input is "every effort moves you", and the LLM
# returns the following logits for the next token:
next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

probas = torch.softmax(next_token_logits, dim=0)


getting the token with the highest probability

In [ ]:
next_token_id = torch.argmax(probas).item()

# The next generated token is then as follows:
print(inverse_vocab[next_token_id])

Function for sampling the next token according the probability

In [ ]:
def print_sampled_tokens(probas):
    torch.manual_seed(123) # Manual seed for reproducibility
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample), minlength=len(probas))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")

Softmax with temperature

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)

# Temperature values
temperatures = [1, 0.1, 5]  # Original, higher confidence, and lower confidence

# Calculate scaled probabilities
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

In [ ]:
for T, probas in zip(temperatures, scaled_probas):
    print(f"\nTemperature: {T}")
    print_sampled_tokens(probas)

looking at the output the word 'pizza' is sampled:
- temperature 1: #0
- temperature 0.1: #0
- temperature 5: #32

In [ ]:
def get_token_sampling_stats(probas, token_id):
    torch.manual_seed(123) # Manual seed for reproducibility
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample), minlength=len(probas))
    #for i, freq in enumerate(sampled_ids):
    #    print(f"{freq} x {inverse_vocab[i]}")

    return sampled_ids[token_id].item()

In [ ]:
for T, probas in zip(temperatures, scaled_probas):
    print(f"\nTemperature: {T}")
    pizza_count = get_token_sampling_stats(probas, vocab['pizza'])
    print(f"Number of times 'pizza' was sampled: {pizza_count}")